# REVE v4: разбор ошибок — где модель ошибается и на что «отвлекается»

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-27 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Источники** | общий набор `zakharsann/h1-prep-shared` (сигнал), выход `zakharsann/h1-reve-v4` (веса `reve_v4_seed{0,1,2}.pt`, `reve_v4_model.py`, оценки окон `reve_v4_window_scores.csv.gz`). Без переобучения |
| **Оценка записи** | ансамбль 3 запусков: вероятность окна = среднее по запускам; запись = среднее верхних 20 % окон, все окна 10 с подряд |
| **Ошибки** | порог — Юден внутри каждого набора (описательный разбор, не оценка качества): ложная тревога — здоровый (норма) выше порога, пропуск — эпилепсия (патология) ниже. Для картинок — здоровые с самыми высокими оценками и больные с самыми низкими |
| **Факторы записи** | амплитуда (СКО), мышечная помеха (доля мощности 20–70 Гц от 0,5–70 Гц), моргания (мощность 0,5–3 Гц в Fp1/Fp2 к остальным каналам), наводка 50 Гц (доля мощности 49–51 Гц; сигнал уже после режекции — это остаток), «мёртвые» каналы (СКО < 0,1 медианы каналов), каналы с чрезмерной амплитудой (СКО > 3 медиан), длина, доля окон с артефактом (мышцы, моргания или всплеск амплитуды выше 95-го процентиля окон отложенных 15 % NMT) |
| **Статистика** | ошибки против верных внутри класса — Манн–Уитни (эффект — вероятность, что у ошибки фактор больше, 95% ДИ бутстреп), связь оценки с фактором внутри класса — Спирмен (95% ДИ бутстреп); поправка Бенджамини–Хохберга |
| **Важность** | окклюзия: в 3 самых подозрительных окнах каждой записи NMT eval и Турции по очереди каждый канал заменяется средним остальных, каждая секунда — средним канала по окну; падение вероятности окна (среднее по 3 запускам) |
| **Исправление** | без переобучения: при подсчёте оценки записи не учитывать окна, где признак артефакта выше порога; признак и порог выбираются по отложенным 15 %, затем NMT eval и Турция |
| **Среда** | Kaggle, T4 |


In [ ]:
"""Общий набор Г1 (NMT + турецкий) — выход блокнота Kaggle zakharsann/h1-prep-shared.

Подключение в kernel-metadata.json: "kernel_sources": ["zakharsann/h1-prep-shared"].
Блокноты моделей копируют этот файл в начало (одна ячейка) и пользуются функциями ниже.

Формат (подробно — в описании блокнота h1_2026-10-26_zs_prep-shared.ipynb):
- h1_shared_meta.csv — по строке на запись: dataset, part, split, id, y, ..., shard200/off200/n200 (и shard256/... если есть);
- h1_shared_info.json — фильтр, множитель масштаба NMT, сколько минут NMT train сохранено, порядок каналов;
- fs200/<имя>.npy — плоские массивы float16: записи (19, n) в C-порядке подряд, запись = arr[off : off + 19*n].
Сигнал в мкВ, 19 каналов в порядке EL, средний референт, 0,1-75 Гц + режекция 50 Гц. Окна НЕ нормированы.
"""
import glob, json, os
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.signal as ss

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]


def find_root(base="/kaggle/input"):
    """Папка общего набора среди подключённых источников."""
    hits = glob.glob(f"{base}/**/h1_shared_meta.csv", recursive=True)
    assert hits, "общий набор не подключён: добавьте zakharsann/h1-prep-shared в kernel_sources"
    return os.path.dirname(hits[0])


def load_meta(root):
    """Таблица записей и описание набора."""
    meta = pd.read_csv(f"{root}/h1_shared_meta.csv")
    info = json.load(open(f"{root}/h1_shared_info.json", encoding="utf-8"))
    return meta, info


_SHARDS = {}
def _shard(root, fs, name):
    k = (root, fs, name)
    if k not in _SHARDS:
        _SHARDS[k] = np.load(f"{root}/fs{fs}/{name}", mmap_mode="r")
    return _SHARDS[k]


def read_record(root, row, fs=200):
    """Одна запись: (19, n) float16, мкВ. row — строка meta (Series или dict).
    Через mmap — для единичных записей; много записей читать через read_all (на /kaggle/input mmap очень медленный)."""
    n, off = int(row[f"n{fs}"]), int(row[f"off{fs}"])
    return np.array(_shard(root, fs, row[f"shard{fs}"])[off:off + 19 * n]).reshape(19, n)


def read_all(root, meta, fs=200, threads=4):
    """Все записи meta в словарь id -> (19, n) float16: каждый файл-кусок читается целиком последовательно, записи режутся в памяти."""
    def load(item):
        name, g = item
        arr = np.load(f"{root}/fs{fs}/{name}")
        return [(i, arr[o:o + 19 * n].reshape(19, n).copy()) for i, o, n in zip(g["id"], g[f"off{fs}"], g[f"n{fs}"])]
    out = {}
    with ThreadPoolExecutor(threads) as ex:
        for part in ex.map(load, list(meta.groupby(f"shard{fs}"))):
            out.update(part)
    return {i: out[i] for i in meta["id"]}


def windows(X, L, S=None):
    """Окна длиной L с шагом S (по умолчанию S = L, подряд): (k, 19, L)."""
    S = S or L
    k = (X.shape[-1] - L) // S + 1 if X.shape[-1] >= L else 0
    if k <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    return np.lib.stride_tricks.sliding_window_view(X, L, axis=-1)[..., ::S, :][..., :k, :].swapaxes(0, -2).copy()


def random_windows(X, L, K, rng):
    """K окон длиной L со случайным началом (любой отсчёт) — свежие окна на каждой эпохе: (K, 19, L)."""
    n = X.shape[-1] - L + 1
    if n <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    st = np.sort(rng.integers(0, n, K))
    return np.stack([X[..., s:s + L] for s in st])


def resample(X, fs_from, fs_to):
    """Пересчёт частоты (полифазный, как scipy.signal.resample_poly) по последней оси, float32 на выходе."""
    if fs_from == fs_to:
        return X
    fr = Fraction(int(fs_to), int(fs_from))
    return ss.resample_poly(np.asarray(X, np.float32), fr.numerator, fr.denominator, axis=-1).astype(np.float32)


def highpass(X, f, fs=200, order=4):
    """Дофильтровка сверху общей нижней границы 0,1 Гц (например, 0,3 Гц как у CBraMod): по всей записи, не по окну."""
    sos = ss.butter(order, f, btype="highpass", fs=fs, output="sos")
    return ss.sosfiltfilt(sos, np.asarray(X, np.float64), axis=-1).astype(np.float32)


In [ ]:
import os, glob, time, json, io
SMOKE = False   # True — короткая проверка на процессоре Kaggle (несколько записей), цифры не смотреть
import numpy as np, pandas as pd, torch, scipy.signal as ss
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import roc_auc_score, roc_curve
from concurrent.futures import ThreadPoolExecutor
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import mne
from PIL import Image
mne.set_log_level("ERROR")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working"; FIG = f"{OUT}/figures"; os.makedirs(FIG, exist_ok=True)
T_NB = time.time()
SFREQ, L, HP = 200, 2000, 0.3
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

SRC = os.path.dirname(sorted(glob.glob("/kaggle/input/**/reve_v4_seed0.pt", recursive=True))[0])
exec(compile(open(f"{SRC}/reve_v4_model.py", encoding="utf-8").read(), "reve_v4_model.py", "exec"))
POS = positions(DEV)
MODELS = []
for seed in [0, 1, 2]:
    ck = torch.load(f"{SRC}/reve_v4_seed{seed}.pt", map_location="cpu", weights_only=False)
    m = RVNet(POS); m.load_state_dict(ck["state_dict"]); MODELS.append(m.to(DEV).eval())
print("веса:", SRC, "| лучшие эпохи:", [torch.load(f"{SRC}/reve_v4_seed{s}.pt", map_location="cpu", weights_only=False)["config"]["best_epoch"] for s in [0, 1, 2]])

def prob(X, bs=96):
    """X: (n, 19, L) мкВ -> вероятность каждого окна, среднее по 3 запускам."""
    out = np.zeros(len(X), np.float64)
    with torch.no_grad(), torch.autocast(**AMP):
        for b in range(0, len(X), bs):
            x = rv_input(X[b:b + bs]).to(DEV)
            out[b:b + bs] = np.mean([torch.sigmoid(m(x).float()).cpu().numpy() for m in MODELS], 0)
    return out

plt.rcParams.update({"font.size": 10, "axes.titlesize": 10, "figure.dpi": 100})
def save_png(fig, name, limit=290_000):
    """PNG до ~300 КБ: если больше — 256 цветов, затем меньше точек на дюйм."""
    path = f"{FIG}/{name}"
    for dpi in [100, 85, 70, 60]:
        buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight"); buf.seek(0)
        img = Image.open(buf)
        if buf.getbuffer().nbytes > limit:
            img = img.convert("RGB").quantize(256)
        img.save(path, optimize=True)
        if os.path.getsize(path) <= limit:
            break
    plt.close(fig)
    return os.path.getsize(path)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")


## 1. Сигнал, оценки окон, ошибки (порог Юдена внутри набора)


In [ ]:
ROOT = find_root()
meta, info = load_meta(ROOT)
wdf = pd.read_csv(f"{SRC}/reve_v4_window_scores.csv.gz")
DS = {"nmt_val": "val", "nmt_eval": "eval", "turkey": "turkey"}
NAME = {"nmt_val": "NMT, отложенные 15 %", "nmt_eval": "NMT eval", "turkey": "Турция"}
CLS = {"nmt_eval": ("норма", "патология"), "turkey": ("здоровые", "эпилепсия"), "nmt_val": ("норма", "патология")}
ids = {ds: wdf[wdf.dataset == ds].id.unique() for ds in DS}
if SMOKE:
    ids = {ds: np.concatenate([wdf[(wdf.dataset == ds) & (wdf.y == c)].id.unique()[:3] for c in [0, 1]]) for ds in DS}
    wdf = wdf[[i in set(ids[d]) for d, i in zip(wdf.dataset, wdf.id)]]
need = pd.concat([meta[(meta.split == s) & meta.id.isin(ids[ds])] for ds, s in DS.items()])
t0 = time.time()
RAW = read_all(ROOT, need)          # 0,1–75 Гц + режекция 50 Гц, мкВ (NMT — с множителем ×100)
print(f"прочитано {len(RAW)} записей за {time.time()-t0:.0f} с")

# окно = среднее вероятностей 3 запусков; запись = среднее верхних 20 % окон (все окна подряд)
top20 = lambda p: np.sort(np.asarray(p))[::-1][:int(np.ceil(0.2 * len(p)))].mean()
W = wdf.groupby(["dataset", "id", "window"], sort=False).agg(y=("y", "first"), p=("p", "mean")).reset_index()
R = W.groupby(["dataset", "id"], sort=False).agg(y=("y", "first"), score=("p", top20), n_win=("p", "size")).reset_index()
THR = {}
for ds in DS:
    d = R[R.dataset == ds]; fpr, tpr, th = roc_curve(d.y, d.score); THR[ds] = float(th[np.argmax(tpr - fpr)])
R["thr"] = R.dataset.map(THR); R["pred"] = (R.score >= R.thr).astype(int)
R["group"] = np.select([(R.y == 1) & (R.pred == 1), (R.y == 0) & (R.pred == 1), (R.y == 1) & (R.pred == 0)], ["TP", "FP", "FN"], "TN")
for ds in DS:
    d = R[R.dataset == ds]
    print(f"{NAME[ds]}: ROC AUC {roc_auc_score(d.y, d.score):.3f}, порог Юдена {THR[ds]:.3f} | " + ", ".join(f"{g} {int((d.group == g).sum())}" for g in ["TP", "FP", "FN", "TN"]))


## 2. Факторы записи и признаки окон


In [ ]:
FR = np.fft.rfftfreq(L, 1 / SFREQ); HANN = np.hanning(L).astype(np.float32)
band = lambda lo, hi: (FR >= lo) & (FR < hi)
B_TOT, B_MUS, B_BLK, B_LINE = band(0.5, 70), band(20, 70), band(0.5, 3), band(49, 51)
FPI, OTH = [0, 1], list(range(2, 19))           # Fp1, Fp2 / остальные

def features(item):
    """Признаки окон 10 с (та же сетка, что у модели) и записи целиком."""
    (ds, i) = item
    X = RAW[i].astype(np.float32); k = X.shape[1] // L
    Wd = X[:, :k * L].reshape(19, k, L).transpose(1, 0, 2)
    Wd = Wd - Wd.mean(-1, keepdims=True)
    P = np.abs(np.fft.rfft(Wd * HANN, axis=-1)) ** 2                       # (окна, 19, частоты)
    tot, mus, blk, line = (P[..., b].sum(-1) for b in [B_TOT, B_MUS, B_BLK, B_LINE])   # (окна, 19)
    ch_w = Wd.std(-1)                                                       # СКО канала в окне
    wf = pd.DataFrame({"dataset": ds, "id": i, "window": np.arange(k),
                       "w_muscle": (mus / (tot + 1e-12)).mean(1),
                       "w_blink": blk[:, FPI].mean(1) / (blk[:, OTH].mean(1) + 1e-12),
                       "w_line": (line / (tot + 1e-12)).mean(1),
                       "w_jump": ch_w.max(1) / (np.median(ch_w) + 1e-9)})
    ch = X.std(1); med = np.median(ch)
    rf = dict(dataset=ds, id=i, amp_uv=float(X.std()), muscle=float(mus.sum() / tot.sum()),
              blink=float(blk[:, FPI].sum() / len(FPI) / (blk[:, OTH].sum() / len(OTH) + 1e-12)),
              line50=float(line.sum() / tot.sum()), n_dead=int((ch < 0.1 * med).sum()), n_extreme=int((ch > 3 * med).sum()),
              dur_min=X.shape[1] / SFREQ / 60)
    return wf, rf

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    res = list(ex.map(features, [(ds, i) for ds in DS for i in ids[ds]]))
WF = pd.concat([r[0] for r in res], ignore_index=True); RF = pd.DataFrame([r[1] for r in res])
# окно с артефактом: мышцы, моргания или всплеск амплитуды выше 95-го процентиля окон отложенных 15 % NMT
ART_Q = {f: float(np.percentile(WF[WF.dataset == "nmt_val"][f], 95)) for f in ["w_muscle", "w_blink", "w_jump"]}
WF["w_art"] = ((WF.w_muscle > ART_Q["w_muscle"]) | (WF.w_blink > ART_Q["w_blink"]) | (WF.w_jump > ART_Q["w_jump"])).astype(int)
RF = RF.merge(WF.groupby(["dataset", "id"]).w_art.mean().rename("art_frac").reset_index(), on=["dataset", "id"])
R = R.merge(RF, on=["dataset", "id"], how="left")
W = W.merge(WF, on=["dataset", "id", "window"], how="left")
assert W.w_muscle.notna().all(), "сетка окон не совпала"
print(f"признаки: {len(WF)} окон, {len(RF)} записей, {time.time()-t0:.0f} с | пороги артефакта окна (95-й процентиль, отложенные NMT):",
      {k: round(v, 3) for k, v in ART_Q.items()})
FACT = {"amp_uv": "амплитуда (СКО), мкВ", "muscle": "мышцы: доля 20–70 Гц", "blink": "моргания: 0,5–3 Гц Fp1/Fp2 к остальным",
        "line50": "наводка 50 Гц: доля 49–51 Гц", "n_dead": "«мёртвые» каналы, шт.", "n_extreme": "каналы с чрезмерной амплитудой, шт.",
        "dur_min": "длина записи, мин", "art_frac": "доля окон с артефактом"}
R.to_csv(f"{OUT}/reve_v4_errors_factors.csv", index=False)
print(R.groupby(["dataset", "group"])[list(FACT)].median().round(3).to_string())


## 3. Ошибки против верных (Манн–Уитни) и связь оценки с факторами внутри класса (Спирмен)


In [ ]:
def bh(p):
    """Поправка Бенджамини–Хохберга (nan пропускаются)."""
    p = np.asarray(p, float); q = np.full_like(p, np.nan); ok = ~np.isnan(p); pv = p[ok]; n = len(pv)
    if n:
        o = np.argsort(pv); r = pv[o] * n / np.arange(1, n + 1); r = np.minimum.accumulate(r[::-1])[::-1]
        q[np.where(ok)[0][o]] = np.minimum(r, 1)
    return q

RNG = np.random.default_rng(0); NB = 300 if SMOKE else 1000
rows = []
for ds in ["nmt_eval", "turkey"]:
    for c in [0, 1]:
        d = R[(R.dataset == ds) & (R.y == c)]
        err = (d.group == ("FP" if c == 0 else "FN")).to_numpy(); s = d.score.to_numpy()
        for f, fname in FACT.items():
            v = d[f].to_numpy(float); a, b = v[err], v[~err]
            row = dict(набор=NAME[ds], класс=CLS[ds][c], ошибка="ложные тревоги" if c == 0 else "пропуски", фактор=fname, key=f,
                       n_ошибок=int(err.sum()), n_верных=int((~err).sum()), медиана_ошибки=np.median(a) if len(a) else np.nan, медиана_верные=np.median(b))
            if len(a) >= 3 and len(b) >= 3 and np.ptp(v) > 0:
                U, p = mannwhitneyu(a, b, alternative="two-sided"); e = U / (len(a) * len(b))
                bs = [(lambda aa, bb: mannwhitneyu(aa, bb).statistic / (len(aa) * len(bb)))(RNG.choice(a, len(a)), RNG.choice(b, len(b))) for _ in range(NB)]
                row.update(MW_эффект=e, MW_ДИ=f"{np.percentile(bs, 2.5):.2f}–{np.percentile(bs, 97.5):.2f}", MW_p=p)
            if np.ptp(v) > 0 and len(v) >= 5:
                rho, p = spearmanr(s, v); bs = []
                for _ in range(NB):
                    j = RNG.integers(0, len(v), len(v))
                    if np.ptp(v[j]) > 0: bs.append(spearmanr(s[j], v[j]).correlation)
                row.update(Спирмен=rho, Спирмен_ДИ=f"{np.nanpercentile(bs, 2.5):+.2f}…{np.nanpercentile(bs, 97.5):+.2f}", Спирмен_p=p)
            rows.append(row)
ST = pd.DataFrame(rows)
for c in ["MW_эффект", "MW_p", "Спирмен", "Спирмен_p"]:
    if c not in ST: ST[c] = np.nan
ST["MW_q"], ST["Спирмен_q"] = bh(ST.MW_p), bh(ST.Спирмен_p)
ST.to_csv(f"{OUT}/reve_v4_errors_stats.csv", index=False)
pd.set_option("display.width", 250)
print(ST.drop(columns=["key"]).round(3).to_string(index=False))

# внутри записи: выше ли оценка окна там, где больше артефакта? (средний Спирмен по записям класса, ДИ — бутстреп по записям)
wrows = []
for ds in ["nmt_eval", "turkey"]:
    for c in [0, 1]:
        d = W[(W.dataset == ds) & (W.y == c)]
        for f in ["w_muscle", "w_blink", "w_jump", "w_line"]:
            r_ = np.array([spearmanr(g.p, g[f]).correlation for _, g in d.groupby("id") if len(g) >= 10 and np.ptp(g[f]) > 0 and np.ptp(g.p) > 0])
            if len(r_) < 3: continue
            bs = [np.mean(RNG.choice(r_, len(r_))) for _ in range(NB)]
            wrows.append(dict(набор=NAME[ds], класс=CLS[ds][c], признак_окна=f, записей=len(r_), средний_Спирмен=r_.mean(),
                              ДИ=f"{np.percentile(bs, 2.5):+.2f}…{np.percentile(bs, 97.5):+.2f}"))
WST = pd.DataFrame(wrows); WST.to_csv(f"{OUT}/reve_v4_errors_window_stats.csv", index=False)
print("\nВнутри записи: Спирмен оценки окна с признаком артефакта окна")
print(WST.round(3).to_string(index=False))

# край записи: как часто самое подозрительное окно — первое или последнее окно записи (там краевой эффект фильтра)
W["edge"] = W.groupby(["dataset", "id"]).window.transform(lambda w: (w == w.min()) | (w == w.max()))
top1 = W.loc[W.groupby(["dataset", "id"]).p.idxmax()]
EDGE = top1.merge(R[["dataset", "id", "group"]], on=["dataset", "id"]).groupby(["dataset", "group"]).edge.agg(["mean", "size"])
EDGE["ожидалось при случайном"] = (2 / R.groupby(["dataset", "group"]).n_win.median()).reindex(EDGE.index)
print("\nДоля записей, у которых самое подозрительное окно — первое или последнее:"); print(EDGE.round(3).to_string())
EDGE.reset_index().to_csv(f"{OUT}/reve_v4_errors_edge.csv", index=False)


## 4. Картинки: факторы, оценка вдоль записи


In [ ]:
GR = ["TN", "FP", "FN", "TP"]
GCOL = {"TN": "#4C78A8", "FP": "#E45756", "FN": "#F58518", "TP": "#54A24B"}
def glab(ds, g):
    h, s = CLS[ds]
    return {"TN": f"{h}:\nверно", "FP": f"{h}:\nложная тревога", "FN": f"{s}:\nпропуск", "TP": f"{s}:\nверно"}[g]
SIZES = {}

# 1. факторы по группам
for n, ds in enumerate(["turkey", "nmt_eval"], 1):
    d = R[R.dataset == ds]
    fig, axs = plt.subplots(2, 4, figsize=(17, 8))
    for ax, (f, fname) in zip(axs.ravel(), FACT.items()):
        vals = [d[d.group == g][f].to_numpy(float) for g in GR]
        ax.boxplot([v if len(v) else [np.nan] for v in vals], widths=0.55, showfliers=False, medianprops=dict(color="k"))
        for j, (g, v) in enumerate(zip(GR, vals)):
            ax.scatter(j + 1 + RNG.uniform(-0.18, 0.18, len(v)), v, s=9, color=GCOL[g], alpha=0.7)
        q = ST[(ST.набор == NAME[ds]) & (ST.key == f)]
        qs = " | ".join(f"{'тревоги' if r.ошибка == 'ложные тревоги' else 'пропуски'}: q={r.MW_q:.2f}" for r in q.itertuples() if not np.isnan(r.MW_q))
        ax.set_title(f"{fname}\n{qs}", fontsize=9)
        ax.set_xticks(range(1, 5)); ax.set_xticklabels([f"{glab(ds, g)}\n(n={len(v)})" for g, v in zip(GR, vals)], fontsize=7.5)
        if f in ["muscle", "blink", "line50", "amp_uv"] and np.nanmin(np.concatenate(vals)) > 0: ax.set_yscale("log")
    fig.suptitle(f"{NAME[ds]}: факторы записи у ошибок и верных случаев (порог Юдена {THR[ds]:.2f}; q — Манн–Уитни, ошибка против верных того же класса, с поправкой)"
                 + ("\nNMT: мкВ приблизительные (масштаб восстановлен множителем ×100)" if ds != "turkey" else ""), fontsize=11)
    fig.tight_layout(); SIZES[f"0{n}_factors_{ds}.png"] = save_png(fig, f"0{n}_factors_{ds}.png")

# 2. Спирмен оценки с фактором внутри класса
cols = [(ds, c) for ds in ["turkey", "nmt_eval"] for c in [0, 1]]
M = np.array([[ST[(ST.набор == NAME[ds]) & (ST.класс == CLS[ds][c]) & (ST.key == f)].Спирмен.mean() for ds, c in cols] for f in FACT])
Q = np.array([[ST[(ST.набор == NAME[ds]) & (ST.класс == CLS[ds][c]) & (ST.key == f)].Спирмен_q.mean() for ds, c in cols] for f in FACT])
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(M, cmap="RdBu_r", vmin=-0.6, vmax=0.6, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, "—" if np.isnan(M[i, j]) else f"{M[i, j]:+.2f}" + ("*" if Q[i, j] < 0.05 else ""), ha="center", va="center", fontsize=10)
ax.set_xticks(range(len(cols))); ax.set_xticklabels([f"{NAME[ds]}:\n{CLS[ds][c]}" for ds, c in cols])
ax.set_yticks(range(len(FACT))); ax.set_yticklabels(list(FACT.values()))
fig.colorbar(im, ax=ax, label="Спирмен: оценка модели и фактор")
ax.set_title("Связь оценки модели с фактором внутри класса (* — q < 0,05 с поправкой)")
fig.tight_layout(); SIZES["03_spearman.png"] = save_png(fig, "03_spearman.png")

# 3. Оценка по окнам вдоль записи
def case_lists(ds):
    d = R[R.dataset == ds]
    fp = d[d.y == 0].sort_values("score", ascending=False); fn = d[d.y == 1].sort_values("score")
    nf, nn_ = max(5, int((fp.group == "FP").sum())), max(5, int((fn.group == "FN").sum()))
    tp = d[d.group == "TP"].sort_values("score", ascending=False); tn = d[d.group == "TN"].sort_values("score")
    return dict(FP=fp.head(min(nf, 12)), FN=fn.head(min(nn_, 12)), TP=tp, TN=tn.iloc[len(tn) // 2 - 1:] if len(tn) > 2 else tn)
CASES = {ds: case_lists(ds) for ds in ["turkey", "nmt_eval"]}
TITLE = {"FP": "ложная тревога", "FN": "пропуск", "TP": "верно: болезнь", "TN": "верно: норма"}

def plot_time(rows, title, fname, errors=True):
    n = len(rows); nc = 3; nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(16, 2.9 * nr), squeeze=False)
    for ax, r in zip(axs.ravel(), rows.itertuples()):
        w = W[(W.dataset == r.dataset) & (W.id == r.id)].sort_values("window")
        t = (w.window.to_numpy() + 0.5) * 10 / 60; p = w.p.to_numpy()
        hi = p >= np.sort(p)[::-1][int(np.ceil(0.2 * len(p))) - 1]
        ax.plot(t, p, color="#555", lw=0.8)
        ax.scatter(t[hi], p[hi], s=10, color=GCOL[r.group], zorder=3, label="верхние 20 % окон")
        a = w.w_art.to_numpy() == 1
        ax.scatter(t[a], np.full(a.sum(), -0.04), marker="|", s=40, color="k", label="окно с артефактом")
        ax.axhline(r.score, ls="--", color=GCOL[r.group], lw=1, label="оценка записи")
        ax.axhline(r.thr, ls=":", color="r", lw=1, label="порог")
        suf = "" if not errors or r.group in ("FP", "FN") else (" (ниже порога)" if r.y == 0 else " (выше порога)")
        ax.set_title(f"{r.id} · оценка {r.score:.2f}{suf}", fontsize=9)
        ax.set_ylim(-0.08, 1.02); ax.set_xlabel("время, мин", fontsize=8); ax.set_ylabel("вероятность окна", fontsize=8)
    for ax in axs.ravel()[n:]: ax.axis("off")
    axs.ravel()[0].legend(fontsize=7, loc="upper right")
    fig.suptitle(title, fontsize=11); fig.tight_layout()
    SIZES[fname] = save_png(fig, fname)

k = 4
for ds in ["turkey", "nmt_eval"]:
    h, s = CLS[ds]
    plot_time(CASES[ds]["FP"], f"{NAME[ds]}: {h} с самыми высокими оценками (ложные тревоги) — оценка каждого окна 10 с вдоль записи", f"0{k}_time_{ds}_false_alarms.png"); k += 1
    plot_time(CASES[ds]["FN"], f"{NAME[ds]}: {s} с самыми низкими оценками (пропуски) — оценка каждого окна 10 с вдоль записи", f"0{k}_time_{ds}_misses.png"); k += 1
corr = pd.concat([CASES[ds][g].head(3) for ds in ["turkey", "nmt_eval"] for g in ["TP", "TN"]])
plot_time(corr, "Верно распознанные (для сравнения): по 3 записи «болезнь» с самыми высокими оценками и 3 типичных «нормы» в каждом наборе", "08_time_correct.png", errors=False)
print({k_: f"{v/1e3:.0f} КБ" for k_, v in SIZES.items()})


## 5. Сигнал в самых подозрительных окнах


In [ ]:
# 4. Сигнал 19 каналов в 2 самых подозрительных окнах (каналы друг под другом, шкала в мкВ)
ORDER = ["Fp1", "F3", "C3", "P3", "O1", "F7", "T3", "T5", "Fz", "Cz", "Pz", "Fp2", "F4", "C4", "P4", "O2", "F8", "T4", "T6"]
OI = [EL.index(c) for c in ORDER]
NICE = [10, 20, 30, 50, 70, 100, 150, 200, 300, 500, 700, 1000, 2000]
HPS = {}
def hp_rec(i):
    if i not in HPS: HPS[i] = highpass(RAW[i], HP)
    return HPS[i]
mmss = lambda s: f"{int(s // 60):02d}:{int(s % 60):02d}"

def art_names(w):
    a = [n for f, n in [("w_muscle", "мышцы"), ("w_blink", "моргание"), ("w_jump", "всплеск амплитуды")] if w[f] > ART_Q[f]]
    return ("артефакт: " + ", ".join(a)) if a else "без артефакта по признакам"

def plot_signal(r, fname, kind):
    w = W[(W.dataset == r.dataset) & (W.id == r.id)].sort_values("p", ascending=False).head(2).sort_values("window")
    X = hp_rec(r.id)
    segs = [X[:, int(k) * L:(int(k) + 1) * L] for k in w.window]
    gap = next((v for v in NICE if v >= 4 * np.median([s.std(1) for s in segs])), NICE[-1])
    fig, axs = plt.subplots(1, len(segs), figsize=(8 * len(segs), 9), squeeze=False)
    for ax, sg, ww in zip(axs[0], segs, w.itertuples()):
        t0 = ww.window * 10; t = t0 + np.arange(L) / SFREQ
        for j, ci in enumerate(OI):
            ax.plot(t, sg[ci] - sg[ci].mean() - j * gap, color="k", lw=0.45)
        ax.set_yticks([-j * gap for j in range(19)]); ax.set_yticklabels(ORDER)
        ax.set_ylim(-19 * gap, gap); ax.set_xlim(t[0], t[-1])
        xt = np.arange(t0, t0 + 11, 2); ax.set_xticks(xt); ax.set_xticklabels([mmss(x) for x in xt])
        ax.set_xlabel("время в записи, мм:сс")
        xb = t[-1] - 0.3; ax.plot([xb, xb], [-18.6 * gap, -17.6 * gap], color="r", lw=2.5)
        ax.text(xb - 0.1, -18.1 * gap, f"{gap} мкВ", color="r", ha="right", va="center", fontsize=9)
        for x in np.arange(t0 + 1, t0 + 10): ax.axvline(x, color="#ddd", lw=0.5, zorder=0)
        ax.set_title(f"окно {mmss(t0)}–{mmss(t0 + 10)} · вероятность окна {ww.p:.2f}\n{art_names(ww._asdict())}", fontsize=10)
    note = "средний референт, 0,3–75 Гц, режекция 50 Гц" + ("; NMT: мкВ приблизительные (масштаб восстановлен ×100)" if r.dataset != "turkey" else "")
    fig.suptitle(f"{NAME[r.dataset]} · {kind_title(kind, r)} · {r.id} · оценка записи {r.score:.2f} (порог {r.thr:.2f})\n"
                 f"два окна 10 с с самой высокой вероятностью; {note}", fontsize=11)
    fig.tight_layout()
    SIZES[fname] = save_png(fig, fname)

def kind_title(kind, r):
    if kind == "FP": return "ложная тревога" if r.group == "FP" else "высокая оценка у здорового (ниже порога)"
    if kind == "FN": return "пропуск" if r.group == "FN" else "низкая оценка у больного (выше порога)"
    return TITLE[kind]

SIG_N = {"FP": 5, "FN": 5, "TP": 2, "TN": 2}
SIG = []
for ds in ["turkey", "nmt_eval"]:
    for g, n in SIG_N.items():
        for j, r in enumerate(CASES[ds][g].head(n).itertuples(), 1):
            fname = f"s_{ds}_{g}_{j}.png"; plot_signal(r, fname, g)
            SIG.append(dict(файл=fname, набор=NAME[ds], тип=g, id=r.id, y=r.y, оценка=round(r.score, 3), группа=r.group))
pd.DataFrame(SIG).to_csv(f"{OUT}/reve_v4_errors_signal_cases.csv", index=False)
print(pd.DataFrame(SIG).to_string(index=False))
print("размеры:", {k: f"{v/1e3:.0f} КБ" for k, v in SIZES.items() if k.startswith("s_")})


## 6. На что смотрит модель: окклюзия каналов и секунд


In [ ]:
# 5. Окклюзия: в 3 самых подозрительных окнах каждой записи закрываем по очереди канал (среднее остальных) и секунду (среднее канала по окну)
def variants(x):
    V = [x]
    for c in range(19):
        z = x.copy(); z[c] = np.delete(x, c, 0).mean(0); V.append(z)
    for s in range(10):
        z = x.copy(); z[:, s * SFREQ:(s + 1) * SFREQ] = x.mean(-1, keepdims=True); V.append(z)
    return np.stack(V)                                    # (30, 19, L)

TOPW = 1 if SMOKE else 3
occ_rows, t0 = [], time.time()
todo = R[R.dataset.isin(["nmt_eval", "turkey"])]
for n, r in enumerate(todo.itertuples()):
    X = hp_rec(r.id).astype(np.float32)
    w = W[(W.dataset == r.dataset) & (W.id == r.id)].sort_values("p", ascending=False).head(TOPW)
    V = np.concatenate([variants(X[:, int(k) * L:(int(k) + 1) * L]) for k in w.window])
    p = prob(V).reshape(len(w), 30)
    for k, pk in zip(w.window, p):
        occ_rows.append(dict(dataset=r.dataset, id=r.id, group=r.group, window=int(k), p_base=pk[0],
                             **{f"ch_{EL[c]}": pk[0] - pk[1 + c] for c in range(19)}, **{f"sec_{s+1}": pk[0] - pk[20 + s] for s in range(10)}))
    if (n + 1) % 50 == 0: print(f"окклюзия {n+1}/{len(todo)} {time.time()-t0:.0f} с", flush=True)
    HPS.pop(r.id, None)
OCC = pd.DataFrame(occ_rows); OCC.to_csv(f"{OUT}/reve_v4_errors_occlusion.csv.gz", index=False, compression="gzip")
CH = [f"ch_{e}" for e in EL]; SEC = [f"sec_{s}" for s in range(1, 11)]
print(f"окклюзия: {len(OCC)} окон, {time.time()-t0:.0f} с")
# среднее по окнам записи, затем по записям группы (каждая запись — один голос)
OR = OCC.groupby(["dataset", "id", "group"])[CH + SEC + ["p_base"]].mean().reset_index()
print(OR.groupby(["dataset", "group"])[CH].mean().mul(100).round(1).to_string())

info_ = mne.create_info(EL, SFREQ, "eeg"); info_.set_montage("standard_1020")
for n, ds in [(9, "turkey"), (10, "nmt_eval")]:
    d = OR[OR.dataset == ds]; groups = ["TP", "FP", "FN", "TN"]
    vals = {g: d[d.group == g][CH].mean().to_numpy() * 100 for g in groups}
    vmax = max(np.nanmax(np.abs(v)) for v in vals.values() if np.isfinite(v).any()) or 1
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.8))
    for ax, g in zip(axs, groups):
        v = vals[g]; ng = int((d.group == g).sum())
        if ng == 0 or not np.isfinite(v).all():
            ax.axis("off"); ax.set_title(f"{glab(ds, g).replace(chr(10), ' ')}: нет записей"); continue
        try:
            im, _ = mne.viz.plot_topomap(v, info_, axes=ax, show=False, cmap="RdBu_r", vlim=(-vmax, vmax), names=EL, sensors=True, contours=0)
        except TypeError:
            im, _ = mne.viz.plot_topomap(v, info_, axes=ax, show=False, cmap="RdBu_r", vmin=-vmax, vmax=vmax, names=EL, show_names=True, contours=0)
        top = np.argsort(v)[::-1][:3]
        ax.set_title(f"{glab(ds, g).replace(chr(10), ' ')} (n={ng})\nсильнее всего: " + ", ".join(f"{EL[i]} {v[i]:+.1f}" for i in top), fontsize=9)
    fig.colorbar(im, ax=axs, shrink=0.75, label="падение вероятности окна, п.п.")
    fig.suptitle(f"{NAME[ds]}: на какие каналы опирается модель — падение вероятности окна (п.п.), если канал закрыть средним остальных "
                 f"(3 самых подозрительных окна записи, среднее по записям группы)", fontsize=10)
    SIZES[f"{n:02d}_topomap_{ds}.png"] = save_png(fig, f"{n:02d}_topomap_{ds}.png")

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, ds in zip(axs, ["turkey", "nmt_eval"]):
    d = OR[OR.dataset == ds]
    for g in ["TP", "FP", "FN", "TN"]:
        x = d[d.group == g][SEC].to_numpy() * 100
        if len(x) == 0: continue
        m, se = x.mean(0), x.std(0, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.zeros(10)
        ax.plot(range(1, 11), m, "-o", color=GCOL[g], label=f"{glab(ds, g).replace(chr(10), ' ')} (n={len(x)})", ms=4)
        ax.fill_between(range(1, 11), m - se, m + se, color=GCOL[g], alpha=0.15)
    ax.axhline(0, color="k", lw=0.5); ax.set_xticks(range(1, 11)); ax.set_xlabel("секунда окна 10 с"); ax.set_title(NAME[ds]); ax.legend(fontsize=8)
axs[0].set_ylabel("падение вероятности окна, п.п.")
fig.suptitle("Важность по времени: падение вероятности окна, если закрыть одну секунду (среднее ± ошибка среднего по записям)", fontsize=10)
fig.tight_layout(); SIZES["11_time_importance.png"] = save_png(fig, "11_time_importance.png")


## 7. Исправление без переобучения: исключать окна с артефактом


In [ ]:
# 6. Исправление без переобучения: не учитывать окна с признаком артефакта выше порога; признак и порог — по отложенным 15 %
MIN_KEEP = 3
def rec_score(d, f=None, t=None):
    """Оценка записи = среднее верхних 20 % окон; окна с признаком > t выкидываются (если осталось < 3 окон — берутся все)."""
    if f is None:
        return d.groupby("id", sort=False).p.agg(top20)
    keep = d[f] <= t
    a = d[keep].groupby("id", sort=False).p.agg(top20); n = d[keep].groupby("id", sort=False).size()
    base = d.groupby("id", sort=False).p.agg(top20)
    return base.where(~base.index.isin(n[n >= MIN_KEEP].index), a.reindex(base.index))

WD = {ds: W[W.dataset == ds] for ds in DS}
YD = {ds: W[W.dataset == ds].groupby("id", sort=False).y.first() for ds in DS}
auc = lambda ds, s: roc_auc_score(YD[ds].loc[s.index], s)
BASE = {ds: rec_score(WD[ds]) for ds in DS}
QS = [50, 70, 80, 90, 95, 98]
grid = []
for f in ["w_muscle", "w_blink", "w_jump", "w_line", "w_art"]:
    ths = [0.5] if f == "w_art" else [float(np.percentile(WD["nmt_val"][f], q)) for q in QS]
    for q, t in zip([None] if f == "w_art" else QS, ths):
        sc = {ds: rec_score(WD[ds], f, t) for ds in DS}
        excl = {ds: float((WD[ds][f] > t).mean()) for ds in DS}
        grid.append(dict(признак=f, процентиль=q, порог=t, **{f"auc_{ds}": auc(ds, sc[ds]) for ds in DS}, **{f"выкинуто_{ds}": excl[ds] for ds in DS}, sc=sc))
G = pd.DataFrame(grid)
base_auc_0 = auc("nmt_val", BASE["nmt_val"])
base_auc = {ds: auc(ds, BASE[ds]) for ds in DS}
print("без исправления:", {NAME[d]: round(v, 3) for d, v in base_auc.items()})
print(G.drop(columns="sc").round(3).to_string(index=False))

def paired(ds, s_new, n=1000, seed=0):
    y = YD[ds].to_numpy(); a = s_new.loc[YD[ds].index].to_numpy(); b = BASE[ds].loc[YD[ds].index].to_numpy()
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    return roc_auc_score(y, a) - roc_auc_score(y, b), np.percentile(v, 2.5), np.percentile(v, 97.5)

# при равном ROC AUC на отложенных — порог, который выкидывает меньше окон; исправление выбирается, только если оно лучше «без исправления» на отложенных
G = G.sort_values(["auc_nmt_val", "выкинуто_nmt_val"], ascending=[False, True], kind="stable")
best_per = G.groupby("признак", sort=False).head(1)
CHOSEN = G.iloc[0] if G.iloc[0].auc_nmt_val > base_auc_0 + 1e-9 else None
fix_rows = []
for r in best_per.itertuples():
    row = dict(признак=r.признак, процентиль=r.процентиль, порог=float(f"{r.порог:.6g}"), выбран=CHOSEN is not None and r.признак == CHOSEN.признак and r.порог == CHOSEN.порог)
    for ds in DS:
        dv, lo, hi = paired(ds, r.sc[ds])
        row.update({f"{NAME[ds]}: ROC AUC": round(auc(ds, r.sc[ds]), 3), f"{NAME[ds]}: Δ (95% ДИ)": f"{dv:+.3f} ({lo:+.3f}…{hi:+.3f})",
                    f"{NAME[ds]}: выкинуто окон": f"{getattr(r, 'выкинуто_' + ds):.0%}", f"_d_{ds}": dv, f"_lo_{ds}": lo, f"_hi_{ds}": hi})
    fix_rows.append(row)
FIX = pd.DataFrame(fix_rows)
FIX.drop(columns=[c for c in FIX if c.startswith("_")]).to_csv(f"{OUT}/reve_v4_errors_fix.csv", index=False)
G.drop(columns="sc").to_csv(f"{OUT}/reve_v4_errors_fix_grid.csv", index=False)
print(f"\nбез исправления: " + " | ".join(f"{NAME[d]} {v:.3f}" for d, v in base_auc.items()))
print("выбрано по отложенным: " + (f"{CHOSEN.признак}, процентиль {CHOSEN.процентиль}, порог {CHOSEN.порог:.6g} (отложенные {CHOSEN.auc_nmt_val:.3f}, без исправления {base_auc['nmt_val']:.3f})"
      if CHOSEN is not None else "ничего — ни один вариант не лучше «без исправления» на отложенных"))
print(FIX.drop(columns=[c for c in FIX if c.startswith("_")]).to_string(index=False))

FNAME = {"w_muscle": "мышцы", "w_blink": "моргания", "w_jump": "всплески амплитуды", "w_line": "наводка 50 Гц", "w_art": "любой артефакт (95-й проц.)"}
fig, ax = plt.subplots(figsize=(11, 4.8))
xs = np.arange(len(FIX)); wdt = 0.27
for j, (ds, col) in enumerate(zip(DS, ["#999", "#4C78A8", "#E45756"])):
    d_, lo, hi = FIX[f"_d_{ds}"].to_numpy(), FIX[f"_lo_{ds}"].to_numpy(), FIX[f"_hi_{ds}"].to_numpy()
    ax.bar(xs + (j - 1) * wdt, d_, wdt, color=col, label=NAME[ds] + (" (выбор порога)" if ds == "nmt_val" else ""))
    ax.errorbar(xs + (j - 1) * wdt, d_, yerr=[d_ - lo, hi - d_], fmt="none", color="k", lw=0.8, capsize=2)
ax.axhline(0, color="k", lw=0.6)
ax.set_xticks(xs); ax.set_xticklabels([f"{FNAME[r.признак]}\n" + (f"выше {int(r.процентиль)}-го проц." if r.процентиль == r.процентиль else "флаг") + (" ★" if r.выбран else "") for r in FIX.itertuples()], fontsize=8.5)
ax.set_ylabel("изменение ROC AUC к «без исправления»"); ax.legend(fontsize=8)
ax.set_title("Исключение окон с артефактом при подсчёте оценки записи (без переобучения): порог для каждого признака выбран по отложенным 15 %; ★ — лучший по отложенным;\n"
             f"95% ДИ — парный бутстреп. Без исправления: " + ", ".join(f"{NAME[d]} {v:.3f}" for d, v in base_auc.items()), fontsize=9)
fig.tight_layout(); SIZES["12_fix.png"] = save_png(fig, "12_fix.png")
print("картинки:", len(os.listdir(FIG)), "| больше 300 КБ:", [k for k, v in SIZES.items() if v > 300_000])
print(f"всё время блокнота: {(time.time()-T_NB)/60:.1f} мин")


## 8. Вывод Kaggle (без служебных предупреждений)

Выводы — отчёт `reports/2026-10-27_reve_errors.md`.

GPU-сеанс `zakharsann/h1-reve-errors` (T4, 9,9 мин):

```
веса: /kaggle/input/notebooks/zakharsann/h1-reve-v4 | лучшие эпохи: [3, 1, 2]
устройство: Tesla T4
прочитано 641 записей за 44 с
NMT, отложенные 15 %: ROC AUC 0.907, порог Юдена 0.813 | TP 40, FP 46, FN 9, TN 240
NMT eval: ROC AUC 0.914, порог Юдена 0.787 | TP 75, FP 17, FN 15, TN 78
Турция: ROC AUC 0.911, порог Юдена 0.340 | TP 40, FP 5, FN 10, TN 66
признаки: 52148 окон, 641 записей, 18 с | пороги артефакта окна (95-й процентиль, отложенные NMT): {'w_muscle': 0.114, 'w_blink': 4.466, 'w_jump': 5.437}
                amp_uv  muscle  blink  line50  n_dead  n_extreme  dur_min  art_frac
dataset  group                                                                     
nmt_eval FN     17.203   0.017  1.029     0.0     0.0        0.0    9.200     0.055
         FP     38.110   0.010  1.290     0.0     0.0        0.0   12.583     0.008
         TN     16.207   0.026  1.710     0.0     0.0        0.0   11.125     0.158
         TP     32.879   0.008  1.107     0.0     0.0        0.0   12.150     0.014
nmt_val  FN     18.911   0.017  1.422     0.0     0.0        0.0   14.567     0.034
         FP     28.518   0.010  1.216     0.0     0.0        0.0   10.558     0.034
         TN     17.183   0.022  1.566     0.0     0.0        0.0   11.267     0.100
         TP     35.569   0.009  1.201     0.0     0.0        0.0   12.508     0.023
turkey   FN     17.135   0.035  0.947     0.0     0.0        0.0   18.250     0.345
         FP     31.043   0.018  2.331     0.0     0.0        0.0   18.750     0.344
         TN     12.672   0.030  2.301     0.0     0.0        0.0   18.750     0.340
         TP     34.228   0.027  0.859     0.0     0.0        0.0   17.375     0.302
   набор     класс         ошибка                                 фактор  n_ошибок  n_верных  медиана_ошибки  медиана_верные  MW_эффект     MW_ДИ  MW_p  Спирмен  Спирмен_ДИ  Спирмен_p  MW_q  Спирмен_q
NMT eval     норма ложные тревоги                   амплитуда (СКО), мкВ        17        78          38.110          16.207      0.870 0.80–0.94 0.000    0.568 +0.41…+0.69      0.000 0.000      0.000
NMT eval     норма ложные тревоги                   мышцы: доля 20–70 Гц        17        78           0.010           0.026      0.133 0.06–0.22 0.000   -0.579 -0.71…-0.39      0.000 0.000      0.000
NMT eval     норма ложные тревоги моргания: 0,5–3 Гц Fp1/Fp2 к остальным        17        78           1.290           1.710      0.354 0.22–0.49 0.060   -0.195 -0.40…+0.02      0.058 0.188      0.148
NMT eval     норма ложные тревоги           наводка 50 Гц: доля 49–51 Гц        17        78           0.000           0.000      0.167 0.07–0.27 0.000   -0.415 -0.57…-0.25      0.000 0.000      0.000
NMT eval     норма ложные тревоги                  «мёртвые» каналы, шт.        17        78           0.000           0.000        NaN       NaN   NaN      NaN         NaN        NaN   NaN        NaN
NMT eval     норма ложные тревоги    каналы с чрезмерной амплитудой, шт.        17        78           0.000           0.000      0.487 0.47–0.50 0.519   -0.011 -0.12…+0.09      0.918 0.772      0.989
NMT eval     норма ложные тревоги                      длина записи, мин        17        78          12.583          11.125      0.568 0.39–0.74 0.382    0.130 -0.08…+0.34      0.211 0.713      0.369
NMT eval     норма ложные тревоги                 доля окон с артефактом        17        78           0.008           0.158      0.151 0.08–0.25 0.000   -0.518 -0.67…-0.32      0.000 0.000      0.000
NMT eval патология       пропуски                   амплитуда (СКО), мкВ        15        75          17.203          32.879      0.295 0.14–0.46 0.013    0.362 +0.13…+0.56      0.000 0.052      0.002
NMT eval патология       пропуски                   мышцы: доля 20–70 Гц        15        75           0.017           0.008      0.748 0.63–0.86 0.003   -0.459 -0.64…-0.25      0.000 0.014      0.000
NMT eval патология       пропуски моргания: 0,5–3 Гц Fp1/Fp2 к остальным        15        75           1.029           1.107      0.508 0.33–0.68 0.931   -0.044 -0.25…+0.18      0.680 0.973      0.866
NMT eval патология       пропуски           наводка 50 Гц: доля 49–51 Гц        15        75           0.000           0.000      0.635 0.46–0.81 0.102   -0.275 -0.48…-0.07      0.009 0.260      0.024
NMT eval патология       пропуски                  «мёртвые» каналы, шт.        15        75           0.000           0.000        NaN       NaN   NaN      NaN         NaN        NaN   NaN        NaN
NMT eval патология       пропуски    каналы с чрезмерной амплитудой, шт.        15        75           0.000           0.000      0.498 0.44–0.58 0.970    0.031 -0.22…+0.26      0.773 0.973      0.941
NMT eval патология       пропуски                      длина записи, мин        15        75           9.200          12.150      0.304 0.17–0.46 0.017    0.167 -0.04…+0.37      0.116 0.061      0.249
NMT eval патология       пропуски                 доля окон с артефактом        15        75           0.055           0.014      0.630 0.49–0.77 0.102   -0.185 -0.39…+0.03      0.080 0.260      0.188
  Турция  здоровые ложные тревоги                   амплитуда (СКО), мкВ         5        66          31.043          12.672      0.585 0.21–0.95 0.537    0.337 +0.08…+0.55      0.004 0.772      0.013
  Турция  здоровые ложные тревоги                   мышцы: доля 20–70 Гц         5        66           0.018           0.030      0.285 0.05–0.54 0.113   -0.473 -0.64…-0.25      0.000 0.264      0.000
  Турция  здоровые ложные тревоги моргания: 0,5–3 Гц Fp1/Fp2 к остальным         5        66           2.331           2.301      0.455 0.24–0.66 0.745    0.027 -0.22…+0.26      0.820 0.970      0.957
  Турция  здоровые ложные тревоги           наводка 50 Гц: доля 49–51 Гц         5        66           0.000           0.000      0.418 0.15–0.69 0.551   -0.062 -0.30…+0.18      0.607 0.772      0.809
  Турция  здоровые ложные тревоги                  «мёртвые» каналы, шт.         5        66           0.000           0.000        NaN       NaN   NaN      NaN         NaN        NaN   NaN        NaN
  Турция  здоровые ложные тревоги    каналы с чрезмерной амплитудой, шт.         5        66           0.000           0.000      0.394 0.34–0.44 0.263    0.004 -0.22…+0.22      0.976 0.525      0.989
  Турция  здоровые ложные тревоги                      длина записи, мин         5        66          18.750          18.750      0.486 0.26–0.69 0.928    0.121 -0.11…+0.35      0.313 0.973      0.477
  Турция  здоровые ложные тревоги                 доля окон с артефактом         5        66           0.344           0.340      0.506 0.32–0.66 0.973   -0.119 -0.34…+0.12      0.323 0.973      0.477
  Турция эпилепсия       пропуски                   амплитуда (СКО), мкВ        10        40          17.135          34.228      0.242 0.12–0.40 0.013    0.518 +0.27…+0.72      0.000 0.052      0.000
  Турция эпилепсия       пропуски                   мышцы: доля 20–70 Гц        10        40           0.035           0.027      0.580 0.41–0.75 0.445   -0.135 -0.39…+0.16      0.350 0.772      0.490
  Турция эпилепсия       пропуски моргания: 0,5–3 Гц Fp1/Fp2 к остальным        10        40           0.947           0.859      0.468 0.26–0.66 0.762   -0.190 -0.45…+0.11      0.185 0.970      0.360
  Турция эпилепсия       пропуски           наводка 50 Гц: доля 49–51 Гц        10        40           0.000           0.000      0.505 0.29–0.71 0.971   -0.002 -0.30…+0.29      0.989 0.973      0.989
  Турция эпилепсия       пропуски                  «мёртвые» каналы, шт.        10        40           0.000           0.000        NaN       NaN   NaN      NaN         NaN        NaN   NaN        NaN
  Турция эпилепсия       пропуски    каналы с чрезмерной амплитудой, шт.        10        40           0.000           0.000      0.595 0.44–0.76 0.220   -0.187 -0.45…+0.11      0.193 0.474      0.360
  Турция эпилепсия       пропуски                      длина записи, мин        10        40          18.250          17.375      0.572 0.42–0.72 0.489   -0.161 -0.47…+0.17      0.265 0.772      0.437
  Турция эпилепсия       пропуски                 доля окон с артефактом        10        40           0.345           0.302      0.480 0.28–0.68 0.856   -0.022 -0.29…+0.24      0.877 0.973      0.982
Внутри записи: Спирмен оценки окна с признаком артефакта окна
   набор     класс признак_окна  записей  средний_Спирмен          ДИ
NMT eval     норма     w_muscle       93           -0.286 -0.34…-0.23
NMT eval     норма      w_blink       93            0.022 -0.03…+0.08
NMT eval     норма       w_jump       93            0.225 +0.16…+0.29
NMT eval     норма       w_line       93           -0.218 -0.27…-0.16
NMT eval патология     w_muscle       90           -0.316 -0.38…-0.24
NMT eval патология      w_blink       90           -0.042 -0.09…+0.01
NMT eval патология       w_jump       90            0.205 +0.13…+0.28
NMT eval патология       w_line       90           -0.227 -0.30…-0.15
  Турция  здоровые     w_muscle       71           -0.405 -0.45…-0.36
  Турция  здоровые      w_blink       71           -0.028 -0.08…+0.02
  Турция  здоровые       w_jump       71            0.506 +0.45…+0.56
  Турция  здоровые       w_line       71           -0.048 -0.08…-0.02
  Турция эпилепсия     w_muscle       50           -0.386 -0.45…-0.32
  Турция эпилепсия      w_blink       50           -0.157 -0.22…-0.09
  Турция эпилепсия       w_jump       50            0.361 +0.30…+0.42
  Турция эпилепсия       w_line       50           -0.071 -0.11…-0.03
Доля записей, у которых самое подозрительное окно — первое или последнее:
                 mean  size  ожидалось при случайном
dataset  group                                      
nmt_eval FN     0.200    15                    0.036
         FP     0.118    17                    0.027
         TN     0.179    78                    0.030
         TP     0.013    75                    0.028
nmt_val  FN     0.000     9                    0.023
         FP     0.043    46                    0.032
         TN     0.179   240                    0.030
         TP     0.025    40                    0.027
turkey   FN     0.200    10                    0.018
         FP     0.000     5                    0.018
         TN     0.152    66                    0.018
         TP     0.100    40                    0.019
{'01_factors_turkey.png': '143 КБ', '02_factors_nmt_eval.png': '164 КБ', '03_spearman.png': '72 КБ', '04_time_turkey_false_alarms.png': '127 КБ', '05_time_turkey_misses.png': '218 КБ', '06_time_nmt_eval_false_alarms.png': '240 КБ', '07_time_nmt_eval_misses.png': '250 КБ', '08_time_correct.png': '231 КБ'}
               файл    набор тип                  id  y  оценка группа
  s_turkey_FP_1.png   Турция  FP    2_Normal (5).mat  0   0.613     FP
  s_turkey_FP_2.png   Турция  FP   2_Normal (54).mat  0   0.435     FP
  s_turkey_FP_3.png   Турция  FP   2_Normal (65).mat  0   0.387     FP
  s_turkey_FP_4.png   Турция  FP    2_Normal (2).mat  0   0.384     FP
  s_turkey_FP_5.png   Турция  FP   2_Normal (35).mat  0   0.368     FP
  s_turkey_FN_1.png   Турция  FN  1_epilepsy (7).mat  1   0.146     FN
  s_turkey_FN_2.png   Турция  FN 1_epilepsy (12).mat  1   0.255     FN
  s_turkey_FN_3.png   Турция  FN 1_epilepsy (44).mat  1   0.257     FN
  s_turkey_FN_4.png   Турция  FN 1_epilepsy (43).mat  1   0.259     FN
  s_turkey_FN_5.png   Турция  FN  1_epilepsy (6).mat  1   0.301     FN
  s_turkey_TP_1.png   Турция  TP 1_epilepsy (18).mat  1   0.847     TP
  s_turkey_TP_2.png   Турция  TP 1_epilepsy (10).mat  1   0.839     TP
  s_turkey_TN_1.png   Турция  TN   2_Normal (36).mat  0   0.241     TN
  s_turkey_TN_2.png   Турция  TN   2_Normal (57).mat  0   0.242     TN
s_nmt_eval_FP_1.png NMT eval  FP         0000044.edf  0   0.944     FP
s_nmt_eval_FP_2.png NMT eval  FP         0001347.edf  0   0.895     FP
s_nmt_eval_FP_3.png NMT eval  FP         0002316.edf  0   0.884     FP
s_nmt_eval_FP_4.png NMT eval  FP         0000274.edf  0   0.881     FP
s_nmt_eval_FP_5.png NMT eval  FP         0002344.edf  0   0.880     FP
s_nmt_eval_FN_1.png NMT eval  FN         0000896.edf  1   0.306     FN
s_nmt_eval_FN_2.png NMT eval  FN         0001583.edf  1   0.576     FN
s_nmt_eval_FN_3.png NMT eval  FN         0000093.edf  1   0.627     FN
s_nmt_eval_FN_4.png NMT eval  FN         0000346.edf  1   0.679     FN
s_nmt_eval_FN_5.png NMT eval  FN         0001703.edf  1   0.680     FN
s_nmt_eval_TP_1.png NMT eval  TP         0000246.edf  1   0.999     TP
s_nmt_eval_TP_2.png NMT eval  TP         0001302.edf  1   0.997     TP
s_nmt_eval_TN_1.png NMT eval  TN         0000074.edf  0   0.529     TN
s_nmt_eval_TN_2.png NMT eval  TN         0001234.edf  0   0.536     TN
размеры: {'s_turkey_FP_1.png': '169 КБ', 's_turkey_FP_2.png': '198 КБ', 's_turkey_FP_3.png': '129 КБ', 's_turkey_FP_4.png': '144 КБ', 's_turkey_FP_5.png': '219 КБ', 's_turkey_FN_1.png': '205 КБ', 's_turkey_FN_2.png': '227 КБ', 's_turkey_FN_3.png': '192 КБ', 's_turkey_FN_4.png': '145 КБ', 's_turkey_FN_5.png': '181 КБ', 's_turkey_TP_1.png': '286 КБ', 's_turkey_TP_2.png': '276 КБ', 's_turkey_TN_1.png': '230 КБ', 's_turkey_TN_2.png': '259 КБ', 's_nmt_eval_FP_1.png': '198 КБ', 's_nmt_eval_FP_2.png': '250 КБ', 's_nmt_eval_FP_3.png': '278 КБ', 's_nmt_eval_FP_4.png': '227 КБ', 's_nmt_eval_FP_5.png': '254 КБ', 's_nmt_eval_FN_1.png': '272 КБ', 's_nmt_eval_FN_2.png': '251 КБ', 's_nmt_eval_FN_3.png': '261 КБ', 's_nmt_eval_FN_4.png': '246 КБ', 's_nmt_eval_FN_5.png': '277 КБ', 's_nmt_eval_TP_1.png': '275 КБ', 's_nmt_eval_TP_2.png': '206 КБ', 's_nmt_eval_TN_1.png': '288 КБ', 's_nmt_eval_TN_2.png': '286 КБ'}
окклюзия: 916 окон, 318 с
                ch_Fp1  ch_Fp2  ch_F3  ch_F4  ch_C3  ch_C4  ch_P3  ch_P4  ch_O1  ch_O2  ch_F7  ch_F8  ch_T3  ch_T4  ch_T5  ch_T6  ch_Fz  ch_Cz  ch_Pz
dataset  group                                                                                                                                       
nmt_eval FN        1.2     5.0   -2.6    3.5   -2.9   -1.2   -1.3   -1.4    4.2   -1.8    2.0    6.0    1.9   -2.1   -0.7    0.3    1.4    0.5   -0.8
         FP        1.7     4.3    0.3    3.2    0.2    1.1   -0.4    0.2    0.2    1.3    1.4    4.7    1.1    0.6    3.3    4.2    2.4    0.3    0.9
         TN       -0.7     4.0   -1.3    3.6   -0.2    1.6   -0.2   -0.1    0.2    1.5   -0.3    3.5   -1.8   -0.3    1.3    5.3    0.5    2.2    0.8
         TP        0.8     2.6    0.4    1.7    0.2    0.3   -0.0    0.2    0.4    0.7    1.0    1.6    1.1    0.5    1.6    1.7    1.4    0.2   -0.1
turkey   FN       -6.3     0.9  -10.8   -5.5   -0.7   -3.0   -2.9   -0.4    1.8    4.2   -6.7   -5.8   -3.4   -3.7   -6.9   -5.1   -0.3   -4.6  -11.1
         FP      -10.1    -5.0  -12.4   -5.9   -6.1    2.5   -6.2   -1.2   -1.2   -0.4   -8.0   -1.8   -5.0   -7.4   -5.8   -8.1    2.4    0.3   -8.5
         TN       -6.7    -1.2  -10.8   -3.9   -3.4   -2.8   -3.8   -0.6    1.4    0.6   -6.6   -3.6   -2.9   -4.7   -4.1   -7.0    1.4   -3.3  -10.5
         TP       -4.7    -0.9   -6.7   -0.1   -3.9   -1.3   -4.3    0.2    0.7    2.7   -5.4   -3.3   -3.8   -3.7   -4.6   -4.8    1.9   -2.8   -7.4
без исправления: {'NMT, отложенные 15 %': np.float64(0.907), 'NMT eval': np.float64(0.914), 'Турция': np.float64(0.911)}
 признак  процентиль  порог  auc_nmt_val  auc_nmt_eval  auc_turkey  выкинуто_nmt_val  выкинуто_nmt_eval  выкинуто_turkey
w_muscle        50.0  0.028        0.906         0.907       0.791             0.500              0.426            0.838
w_muscle        70.0  0.045        0.905         0.911       0.823             0.300              0.256            0.648
w_muscle        80.0  0.059        0.906         0.915       0.865             0.200              0.182            0.506
w_muscle        90.0  0.085        0.906         0.914       0.899             0.100              0.092            0.314
w_muscle        95.0  0.114        0.906         0.915       0.914             0.050              0.043            0.176
w_muscle        98.0  0.150        0.907         0.915       0.914             0.020              0.018            0.077
 w_blink        50.0  1.202        0.897         0.904       0.886             0.500              0.493            0.558
 w_blink        70.0  1.792        0.904         0.910       0.884             0.300              0.280            0.397
 w_blink        80.0  2.348        0.906         0.913       0.888             0.200              0.181            0.319
 w_blink        90.0  3.381        0.907         0.913       0.906             0.100              0.087            0.231
 w_blink        95.0  4.466        0.908         0.914       0.914             0.050              0.040            0.175
 w_blink        98.0  5.962        0.907         0.914       0.921             0.020              0.013            0.128
  w_jump        50.0  1.774        0.896         0.887       0.925             0.500              0.507            0.659
  w_jump        70.0  2.265        0.897         0.892       0.938             0.300              0.309            0.345
  w_jump        80.0  2.769        0.901         0.895       0.942             0.200              0.209            0.201
  w_jump        90.0  3.993        0.901         0.907       0.934             0.100              0.098            0.107
  w_jump        95.0  5.437        0.905         0.910       0.924             0.050              0.050            0.068
  w_jump        98.0  7.774        0.907         0.912       0.922             0.020              0.023            0.043
  w_line        50.0  0.000        0.909         0.907       0.911             0.500              0.420            0.101
  w_line        70.0  0.000        0.909         0.908       0.909             0.300              0.249            0.035
  w_line        80.0  0.000        0.909         0.910       0.909             0.200              0.170            0.017
  w_line        90.0  0.000        0.907         0.908       0.910             0.100              0.096            0.004
  w_line        95.0  0.000        0.907         0.914       0.911             0.050              0.056            0.001
  w_line        98.0  0.000        0.908         0.913       0.911             0.020              0.025            0.000
   w_art         NaN  0.500        0.906         0.910       0.926             0.142              0.127            0.396
без исправления: NMT, отложенные 15 % 0.907 | NMT eval 0.914 | Турция 0.911
выбрано по отложенным: w_line, процентиль 50.0, порог 2.14969e-07 (отложенные 0.909, без исправления 0.907)
 признак  процентиль        порог  выбран  NMT, отложенные 15 %: ROC AUC NMT, отложенные 15 %: Δ (95% ДИ) NMT, отложенные 15 %: выкинуто окон  NMT eval: ROC AUC   NMT eval: Δ (95% ДИ) NMT eval: выкинуто окон  Турция: ROC AUC     Турция: Δ (95% ДИ) Турция: выкинуто окон
  w_line        50.0 2.149690e-07    True                          0.909           +0.002 (-0.005…+0.010)                                 50%              0.907 -0.007 (-0.018…+0.002)                     42%            0.911 -0.000 (-0.009…+0.010)                   10%
 w_blink        95.0 4.466160e+00   False                          0.908           +0.001 (-0.001…+0.003)                                  5%              0.914 +0.000 (-0.001…+0.002)                      4%            0.914 +0.003 (-0.010…+0.015)                   17%
  w_jump        98.0 7.774440e+00   False                          0.907           +0.000 (-0.002…+0.003)                                  2%              0.912 -0.002 (-0.004…+0.000)                      2%            0.922 +0.010 (-0.009…+0.032)                    4%
w_muscle        98.0 1.504330e-01   False                          0.907           -0.000 (-0.001…+0.000)                                  2%              0.915 +0.001 (-0.000…+0.002)                      2%            0.914 +0.003 (-0.006…+0.011)                    8%
   w_art         NaN 5.000000e-01   False                          0.906           -0.001 (-0.008…+0.005)                                 14%              0.910 -0.004 (-0.010…+0.001)                     13%            0.926 +0.014 (-0.016…+0.046)                   40%
картинки: 40 | больше 300 КБ: []
всё время блокнота: 9.9 мин
```